In [6]:
# ============================================================
# Imports
# ============================================================

import pkg_resources
import subprocess
from pathlib import Path
import pandas as pd

try:
    from handle_seqrepo import (
        load_SeqRepo,
        get_repofiles,
        get_clean_seq,
        write_fasta,
        mk_ClusterTable,
    )
except ModuleNotFoundError:
    import sys

    sys.path.append("../../utils")

    from handle_seqrepo import (
        load_SeqRepo,
        get_repofiles,
        get_clean_seq,
        write_fasta,
        mk_ClusterTable,
    )

#--------- config -----------
OUTPUT_DIR = Path("../../../data/manuscript/dysgalactiae")

REPO_NAME = "dysgalactiae_repo"

SEQREPO_DIR = Path( "../../../data/SeqRepo/dysgalactiae")
CDHIT_CUTOFFS = [99, 95, 90, 85, 80, 75,70, 65, 60]
#----------------------------

fasta_file = OUTPUT_DIR / f"{REPO_NAME}.fasta"

def run_cdhit_analysis(
    fasta_file,
    repo_frame,
    output_dir,
    seqrepo,
    cutoffs,
):

    for cutoff in cutoffs:

        cutoff_dir = output_dir / f"cdhit_{cutoff}"
        cutoff_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        output_fasta = (
            cutoff_dir /
            f"dysgalactiae_cdhit_{cutoff}.fasta"
        )

        log_file = (
            cutoff_dir /
            f"dysgalactiae_cdhit_{cutoff}.log"
        )
        if cutoff > 60:
            n = 5
        else:
            n = 4
        command = [
            "cd-hit",
            "-i", str(fasta_file),
            "-o", str(output_fasta),
            "-g", "1",
            "-c", str(cutoff / 100),
            "-d", "100",
            "-n", str(n),
        ]

        print(f"Running CD-HIT at {cutoff}% identity...")

        with open(log_file, "w") as log:
            subprocess.run(
                command,
                stdout=log,
                stderr=subprocess.STDOUT,
                check=True
            )

        cluster_file = Path(
            f"{output_fasta}.clstr"
        )

        mk_ClusterTable(
            str(cutoff),
            str(cluster_file),
            str(repo_frame),
            seqrepo
        )

In [2]:
sr=load_SeqRepo(str(SEQREPO_DIR))
repo_frame =  OUTPUT_DIR / f"{REPO_NAME}_frame.csv"

run_cdhit_analysis(
    fasta_file=fasta_file,
    repo_frame=repo_frame,
    output_dir=OUTPUT_DIR,
    seqrepo=sr,
    cutoffs=CDHIT_CUTOFFS
)

Running CD-HIT at 75% identity...
